## Preparação e Tratamento de Dados

**Objetivo:** transformar os arquivos brutos (`funcionarios`, `departamentos`, `filiais`) em uma única tabela analítica, `rh_dados_limpos`, com **uma linha por colaborador**, pronta para os testes de hipótese do notebook `02_analise.ipynb`.

**Fluxo:** `data/raw/*.csv` → tabelas no DuckDB (raw) → diagnóstico → `sql/limpeza.sql` → validação → `data/processed/rh_dados_limpos.{csv,parquet}`

### Decisão técnica: SQL com DuckDB

Para a etapa de tratamento, padronização e criação de variáveis analíticas, optei por utilizar a linguagem **SQL via DuckDB**. Essa decisão técnica foi pautada por três motivos principais:

* **Leitura direta de arquivos:** `read_csv_auto` carrega os CSVs sem etapas intermediárias.
* **Auditabilidade/Transparência:** a lógica de transformação fica em `sql/limpeza.sql`, versionado com o projeto, separada da narrativa do notebook.
* **Escalabilidade:** o DuckDB é orientado a colunas (OLAP). Com o volume atual o ganho de desempenho é irrelevante, mas o mesmo pipeline funcionaria com bases muito maiores.


### Preparação do ambiente - Carga de Dados Brutos

Os três arquivos `.csv` são carregados no DuckDB (`data/rh_dados.duckdb`) **sem nenhuma alteração**, para que todo o tratamento posterior seja feito em SQL e a origem fique preservada.

**Objetivo:** Criar a camada de staging para permitir todo o processo de preparação seja executado a partir da linguagem **SQL**.

In [ ]:
import pandas as pd
import duckdb

con = duckdb.connect('../data/rh_dados.duckdb')

con.execute("""
    CREATE OR REPLACE TABLE funcionarios AS 
    SELECT * FROM read_csv_auto('../data/raw/funcionarios.csv');
""")

con.execute("""
    CREATE OR REPLACE TABLE departamentos AS 
    SELECT * FROM read_csv_auto('../data/raw/departamentos.csv');
""")

con.execute("""
    CREATE OR REPLACE TABLE filiais AS 
    SELECT * FROM read_csv_auto('../data/raw/filiais.csv');
""")


### Visualização para Exploração de Dados

Antes de limpar, as inconsistências foram mapeadas com queries **SQL**. O que foi encontrado:

| Problema | Coluna(s) | 
|---|---|
| Rótulo inconsistente (`Fem`) | `genero` |
| Datas no formato DD/MM/AAAA | `data_admissao`, `data_promocao` | 
| Salário como texto com prefixo `R$ ` e vírgula decimal | `salario_base` | 
| Coluna `nome`, irrelevante para a análise | `nome` | 

A fim de garantir a integridade da base, os dados devem ser verificados contra registros duplicados também. 

In [ ]:
df_sujo = con.execute("""
SELECT * FROM funcionarios
""").df()

pd.set_option('display.max_rows', 10)
df_sujo

### Limpeza, Engenharia de Variáveis e Criação da Tabela Analítica (`sql/limpeza.sql`)

A preparação dos dados foi estruturada com os seguintes focos:

* **Anonimização:** Remoção de dados sensíveis ou irrelevantes para a agregação (ex: nomes).
* **Padronização:** Padronização de textos, tratamento de tipos e de formatação (Ex: datas) e limpeza de strings numéricas (ex: retirada de prefixo de moeda).
* **Engenharia de Variáveis (Feature Engineering):** Criação de colunas binárias (flags) para facilitar a aplicação de testes estatísticos posteriores (ex: `desligado`, `promocao`, `bonus`) e cálculo da remuneração total.

Para manter o notebook visualmente limpo, a query SQL responsável por criar a tabela final unificada e tratada foi isolada no arquivo `sql/limpeza.sql`. No padrão analítico que estamos adotando, a query irá consultar os DataFrames e gerar um único DataFrame final limpo, `rh_dados_limpos`.

In [ ]:
with open('../sql/limpeza.sql', 'r', encoding='utf-8') as f:
    query_limpeza = f.read()

con.execute(query_limpeza)

df = con.execute("SELECT * FROM rh_dados_limpos").df()
pd.set_option('display.max_columns', None)
df

### Exportação

Exportação dos dados limpos em formado `.csv` e  `parquet`.

Os dados foram exportados em dois formatos distintos, visando diferentes cenários de consumo:

* **Formato CSV:** (`;`, `utf-8-sig`) abre corretamente no Excel em português e é fácil de inspecionar.
* **Formato Parquet:** formato colunar compactado que preserva os tipos (datas, inteiros, decimais). O uso do Parquet é o padrão na indústria de dados e garante uma ingestão muito mais eficiente e segura na ferramenta de visualização (PowerBI e Streamlit).

In [ ]:
df.to_csv('../data/processed/rh_dados_limpos.csv', index=False, sep=';', decimal='.', encoding='utf-8-sig')

df.to_parquet('../data/processed/rh_dados_limpos.parquet', index=False, engine='pyarrow')

print(f"Total de registros exportados: {len(df)}")
con.close()